# [ICM Development](@id Development)

The model from [Saiz et al.](https://elifesciences.org/articles/56079) has three parts in the model

## Definition of the model

### Mechanics

The cells are spheroids that behave under the following equations:

$$m_i\frac{dv_i}{dt} =-bv_i+\sum_j F_{ij}$$

$$\frac{dx_i}{dt} =v_i$$

where the force is

$$F_{ij}=
\begin{cases}
F_0(\frac{r_{ij}}{d_{ij}}-1)(\frac{\mu r_{ij}}{d_{ij}}-1)\frac{(x_i-x_j)}{d_{ij}}\hspace{1cm}if\;d_{ij}<\mu r_{ij}\\
0\hspace{5cm}otherwise
\end{cases}$$

where $d_{ij}$ is the Euclidean distance and $r_{ij}$ is the sum of both radius.

### Biochemical interaction

Each cell has a biochemical component that follows an equation of the form:

$$\frac{dx_i}{dt}=\frac{α(1+x^n_i)^m}{(1+x^n_i)^m+(1+(\langle x\rangle_i)/K)^{2m}}-x_i$$

This is similar to the above case. The only detail required is to note that the average expression can be modeled as the combination of two interacting variables. The biochemical system is activated in the interval $[t_{on},t_{off}]$.

We made explicit that the average operator can be written as two interaction parameters that are the contraction along the second index that runs over the neighbours of each cell as,

$$N_{ij}=
\begin{cases}
1\hspace{1cm}d<f_{range}r_{ij}\\
0\hspace{1cm}otherwise
\end{cases}$$

$$X_{ij}=
\begin{cases}
x_j\hspace{1cm}d<f_{range}r_{ij}\\
0\hspace{1cm}otherwise
\end{cases}$$

$$\langle x\rangle_i=\frac{\sum_j X_{ij}}{\sum_j N_{ij}}=\frac{X_{i}}{N_{i}}$$

### Growth

The cells present division. The rules for the division in this model are. Random election of a division direction over the unit sphere. The daughter cells divide equally in mass and volume and are positioned in oposite directions around the division axis centered at the parent cell. The chemical concentration is divided asymmetrically with each cell taking $1\pm\sigma_x \text{Uniform}(0,1)$ for the parent cell. A new division time is assigned to each aghter cell from a uniform distribution $\text{Uniform}(\tau_{div}(1-\sigma_{div}),\tau_{div}(1+\sigma_{div}))$.

## Creation of the Agent

In [ ]:
#Package
using CellBasedModels
#Functions for generating random distributions
using Random
using Distributions
#Package for plotting in 3D
using GLMakie
#using CairoMakie #<-- Nicer but much slower!

# To reset to GLMakie:
GLMakie.activate!()

Makie.inline!(true)

#using MathTexEngine
#Makie.update_theme!(fonts = (regular = texfont(), bold = texfont(:bold), italic = texfont(:italic)))

using CSV
#using DataFrames
import Pkg; Pkg.add("Distances")
import Pkg; Pkg.add("Clustering")
import Pkg; Pkg.add("StaticArrays")
import Pkg; Pkg.add("NearestNeighbors")
using Pkg
Pkg.add("WriteVTK")
using DataFrames, Distances, Clustering
using GeometryBasics
using Colors
using StaticArrays
using Statistics


using Dates

### Define the agent

First, we have to create an instance of an agent with all the propoerties of the agents.First, we have to create an instance of an agent with all the propoerties of the agents.

In [ ]:

#Nmax = 1000
#P_track = [3.2, 5.02]
#P_track_test = [4.44, 6.66]
#pop!(P_track_test)
#pop!(P_track_test)

#P_state = [0]
#pop!(P_state)

model = ABM(3,

    #Inherit model mechanics
    ###baseModelInit = [CBMModels.softSpheres3D],

   
    #Global parameters
    model = Dict(

        
        :b=>Float64,
        :λ=>Float64,
        :ζ=>Float64,
        :μ=>Float64,
        :f0=>Array{Float64},
        :f0_rep=>Array{Float64},
        
        #############################################################################################################################
        #:ndist => Float64, #Maximum distance to compute neighbor distance
        :kpON=>Float64, #Protrusion rate/"probability" (Could make this different for each cell AND change with time!)
        :kpOFF=>Float64, #Protrusion duration rate/"probability" (Could make this different for each cell AND change with time!)
        #:kpOFF_C=>Float64,
        :kpON_=>Array{Float64},
        :kpOFF_=>Array{Float64},
        :fRange_P=>Float64, # Protrusion force interaction range factor
        :fPI=>Float64, # Contractile Protrusion force magnitude
        :ε=>Float64, #LJ minimum energy for cell-cell protrusion interaction
        #:P_track=>Array{Float64}, #<--- EMINA THAME!
        #:PP=>Array{Int64},
        :p=>Float64,  #Transition rate from A to B
        :q=>Float64,  #Transition rate from B to C
        #:κ0=>Float64, #Transition rate feedback factor constant
        :κ0=>Array{Float64}, #Transition rate feedback factor constant
        #:κ=>Array{Float64}, #Transition rate feedback factor
        :fs0=>Float64, #Soft Core repulsion between B and C type cells
        :fs=>Array{Int64},
        :f0_0=>Float64, #Hard Core repulsion between  cells
        :fPI_=>Array{Float64}, # Contractile Protrusion force magnitude

        #Division constants:
        :τDiv_=>Array{Float64},
        :τDiv_pre_=>Array{Float64},
        :σDiv_=>Array{Float64},

        # velocity dissipation:
        :vel_diss=>Float64,
        #############################################################################################################################
        
        #Chemical constants
        :α=>Float64, 
        :K=>Float64, 
        :nn=>Float64, 
        :mm=>Float64,
        #Physical constants
        :fRange=>Float64,
        :mi=>Float64, 
        :ri=>Float64, 
        :k0=>Float64,
        #Division constants
        :fAdh=>Float64, 
        :τDiv=>Float64, 
        :σDiv=>Float64, 
        :c0=>Float64, 
        :σc=>Float64, 
        :nCirc=>Float64, 
        :σNCirc=>Float64,
        :fMin=>Float64, 
        :fMax=>Float64, 
        :fPrE=>Float64, 
        :fEPI=>Float64, 
        :τCirc=>Float64, 
        :στCirc=>Float64, 
        :rESC=>Float64,
        :nOn=>Float64, 
        :cMax=>Float64
    ),

    
    #Local float parameters
    agent = Dict(

        :m=>Float64,
        :r=>Float64,
        :vx=>Float64,
        :vy=>Float64,
        :vz=>Float64,
        :fx=>Float64,
        :fy=>Float64,
        :fz=>Float64,
        
        #############################################################################################################################
        :fpx=>Float64,
        :fpy=>Float64,
        :fpz=>Float64,
        
        :nNeighs=>Int64,  #Number of neighbors that the agent has for velocity term in forces
        :nNeighs_A=>Int64,  #Number of neighbors that the agent has in state A (cellFate = 1)
        :nNeighs_B=>Int64,  #Number of neighbors that the agent has in state B (cellFate = 2)
        :nNeighs_C=>Int64,  #Number of neighbors that the agent has in state C (cellFate = 3)
        #:nNeighs_ABC=>Int64, #Number of neighbors that the agent has in state A,B,C (cellFate = 1,2,3) # agent class cannot have arrays :(
        :nNeighs_P=>Int64,  #Number of neighbours that the agent has for Protrusion term in forces
        :nSumVx=>Float64, #Sum of neighbors' vx
        :nSumVy=>Float64, #Sum of neighbors' vy
        :nSumVz=>Float64, #Sum of neighbors' vz

        :kp=>Float64,  #Cell protrusion random number
        :τp=>Float64,  #Protrusion time constant
        :tij=>Float64, #Protrusion time 
        :tij_c=>Float64, #Protrusion time counter
        #:P=>Int64, #Cell protrusion state (active/inactive)
        #:PP=>Array{Float64},
        #:kp=>Float64,   #Cell protrusion probability

        #For Various aggregates:
        :cell_aggregate=>Int64,
        #For controlling division:
        :cell_dividing=>Int64,
        :cell_div_relax_steps=>Int64,

        :salt_and_pepper_switch=>Int64,

        :N_start_sim_switch=>Int64,
        #############################################################################################################################
        
        :c=>Float64,
        :tDivision=>Float64, #Variable storing the time of division of the cell
        :ci=>Float64, #Chemical activity of the neighbors
        :ni=>Float64,  #Number of neighbors
        :tOff=>Bool,    #indicate if the circuit for that cell is on or off (0,1)
        :cellFate=>Int64 #Identity of the cell (1 A, 2 B, 3 C)
    ),


    
    ###Mechnical & Chemical dynamics
    agentODE = quote
        
        #############################################################################################################################
        #Mechanics
        #vx_[i1_] = 0.0; vy_[i1_] = 0.0; vz_[i1_] = 0.0
        fx = 0.0; fy = 0.0; fz = 0.0
        #fmax = 110.0  # corresponds to a distance of R/3 between the cells
        #fmax = 1482.0  # corresponds to a distance of R/10 between the cells
        fmax = 1000.0  # corresponds to a distance of R/8 between the cells
        switch_fmax_x = 0
        switch_fmax_y = 0
        switch_fmax_z = 0
        @loopOverNeighbors it2 begin
            dij = sqrt((x-x[it2])^2+(y-y[it2])^2+(z-z[it2])^2)
            rij = r+r[it2]
            
            #if dij < μ*rij && dij > 0   
            if dij < μ*rij && dij > 0
                
                if dij < rij

                    fx += ( f0_0 * f0_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(μ*rij/dij-1) ) * (x-x[it2])/dij 
                    fy += ( f0_0 * f0_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(μ*rij/dij-1) ) * (y-y[it2])/dij
                    fz += ( f0_0 * f0_rep[cellFate[i1_],cellFate[it2]] * (rij/dij-1)*(μ*rij/dij-1) ) * (z-z[it2])/dij

                    # IF THIS HAPPENS, MAYBE THERE IS A PROBLEM!
                    # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end
                
                else

                    fx += f0[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(μ*rij/dij-1)*(x-x[it2])/dij
                    fy += f0[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(μ*rij/dij-1)*(y-y[it2])/dij
                    fz += f0[cellFate[i1_],cellFate[it2]]*(rij/dij-1)*(μ*rij/dij-1)*(z-z[it2])/dij

                    # IF THIS HAPPENS, MAYBE THERE IS A PROBLEM!
                    # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end

                end
                
            end

            
        end

       

        
        
        if nNeighs < N_relV_min
       
            dt(x) = fx/λ + fpx/λ
            dt(y) = fy/λ + fpy/λ
            dt(z) = fz/λ + fpz/λ

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
            
        else
            
            dt(x) = fx/nNeighs/λ + nSumVx/nNeighs/ζ + fpx/nNeighs/λ
            dt(y) = fy/nNeighs/λ + nSumVy/nNeighs/ζ + fpy/nNeighs/λ
            dt(z) = fz/nNeighs/λ + nSumVz/nNeighs/ζ + fpz/nNeighs/λ 

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
            
        end

        compile=false
        #############################################################################################################################

    end,
    
    agentRule=quote

        # If we want to list the nNeighs:
        nNeighs_list = zeros(Int64, 0)
        
        nNeighs_P_list = zeros(Int64, 0)
            
        #############################################################################################################################
        fpx = 0.; fpy = 0.; fpz = 0.
        # Count neighbours for each cell & Add neighbour velocities for primary cell velocity calculation
        nNeighs_new = 0 #Set it to zero before starting the computation
        #nNeighs_new_switch = 0
        nNeighs_A_new = 0 #Set it to zero before starting the computation
        nNeighs_B_new = 0 #Set it to zero before starting the computation
        nNeighs_C_new = 0 #Set it to zero before starting the computation
        nNeighs_P_new = 0 #Set it to zero before starting the computation
        nSumVx_new = 0. #Set it to zero before starting the computation
        nSumVy_new = 0. #Set it to zero before starting the computation
        nSumVz_new = 0. #Set it to zero before starting the computation
        nSumV_max = 100.0 #max vel force
        switch_nSumVx_max = 0
        switch_nSumVy_max = 0
        switch_nSumVz_max = 0
        halt_division = 0

        
        
        # FOR INITIALISING SALT & PEPPER DISTRIBUTION: 
        if N == N_start_sim && salt_and_pepper == 1 && salt_and_pepper_switch == 0

            if i1_ in selected_indices
                cellFate = 2
            end
            salt_and_pepper_switch = 1
        end
        
        
        @loopOverNeighbors it2 begin
           
            d = CBMMetrics.euclidean(x,x[it2],y,y[it2],z,z[it2]) #Using euclidean matric provided in package
            ndist = (r + r[it2])*fRange / 1.2 * 1.0 #* 2.0
          
            # For Cell Velocity update:
            if d < ndist
                
                nNeighs_new += 1 #Add 1 to neighbors of cell

                if cellFate[it2] == 1
                    nNeighs_A_new += 1 #Add 1 to A-neighbors of cell
                elseif cellFate[it2] == 2
                    nNeighs_B_new += 1 #Add 1 to B-neighbors of cell
                elseif cellFate[it2] == 3
                    nNeighs_C_new += 1 #Add 1 to C-neighbors of cell
                end


                if cell_dividing == 0 && cell_dividing[it2] == 0
                    
                    nSumVx_new += vx[it2]*vel_diss
                    nSumVy_new += vy[it2]*vel_diss
                    nSumVz_new += vz[it2]*vel_diss
                    
                elseif cell_dividing > 0
                    cell_dividing += 1
                    if cell_dividing > cell_div_relax_steps
                        cell_dividing = 0
                    end
                end

                
                # If we want to list the nNeighs: 
                push!(nNeighs_list, it2)

            end

            # For Protrusion Pair Interaction:
            ndist_P = r*fRange_P   # this is assuming all cells have the same radius! This gives FCC 3rd NNs! (Just before cells are allowed to pull other cells exactly behind their 1st NN (d=4r))
           
            if d < ndist_P && d > μ*2*r
                nNeighs_P_new += 1

                push!(nNeighs_P_list, it2)

            end
        
        end

        

        
        # For Protrusion Force Activation:

        if nNeighs_P_new != 0
            random_neigh = Int64(ceil(CBMDistributions.uniform(0,1) * length(nNeighs_P_list)))
            d = CBMMetrics.euclidean(x,x[random_neigh],y,y[random_neigh],z,z[random_neigh])
        end
       
        # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
        if nNeighs_P_new != 0 && kp < (kpON_[cellFate[i1_],cellFate[random_neigh]]*dt) && PI[i1_] == 0 && d < ndist_P && d >= 2*r && N >= N_prot_min && prot_ON == 1

            tij_new = τp - log(CBMDistributions.uniform(0,1)) / kpOFF_[cellFate[i1_],cellFate[random_neigh]]

            # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            tij = tij_new
            tij_c_new = 0.0
            tij_c = tij_c_new

            # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            PI[i1_] = random_neigh
            append!(PI_list[random_neigh], [i1_])
            
        else
            kp_new = CBMDistributions.uniform(0,1)
            kp = kp_new
        end


        # Finally, update nNeighs and Neighbour Velocity Sums for cell:

        nNeighs = nNeighs_new
        nNeighs_A = nNeighs_A_new
        nNeighs_B = nNeighs_B_new
        nNeighs_C = nNeighs_C_new
       
        nSumVx = nSumVx_new
        nSumVy = nSumVy_new
        nSumVz = nSumVz_new
 
        nNeighs_P = nNeighs_P_new

        
        # Protrusion Force Profile:
        if (PI[i1_] != 0) && (N >= N_prot_min) && prot_ON == 1
            cell_partner = abs(PI[i1_])
           
            d_partner = CBMMetrics.euclidean(x,x[cell_partner],y,y[cell_partner],z,z[cell_partner])
    
            ndist_P = r * fRange_P # Assuming FCC structure with all particles having same radius! 
            
            # Active cell-cell protrusion force interaction between the current cell and its Pair-cell given by abs(P):
            if tij_c < tij && d_partner < ndist_P && d_partner >= 2*r
                
                tij_c += dt

                fpx -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (x-x[cell_partner])/d_partner
                fpy -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (y-y[cell_partner])/d_partner
                fpz -= fPI_[cellFate[i1_],cellFate[cell_partner]] * (z-z[cell_partner])/d_partner
            
                
            elseif tij_c >= tij || d_partner >= ndist_P || d_partner < 2*r
           
                PI[i1_] = 0
                tij_c_new = 0.0
                tij_c = tij_c_new
                kp_new = CBMDistributions.uniform(0,1)
                kp = kp_new

                idx = findfirst(==(i1_), PI_list[cell_partner])
                if idx !== nothing
                    deleteat!(PI_list[cell_partner], idx)
                end
                
            end
        end

        
        # Compute force due to cell being bonded to other cells that extended prots to it:
        
        if (N >= N_prot_min) && prot_ON == 1
            for cell_partnered_to in PI_list[i1_]
                    
                d = CBMMetrics.euclidean(x,x[cell_partnered_to],y,y[cell_partnered_to],z,z[cell_partnered_to])
                ndist_P = r * fRange_P # Assuming FCC structure with all particles having same radius!
        
                if tij_c[cell_partnered_to] < tij[cell_partnered_to] && d != 0 && d < ndist_P && d >= 2*r
                
                    fpx -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (x-x[cell_partnered_to])/d
                    fpy -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (y-y[cell_partnered_to])/d
                    fpz -= fPI_[cellFate[cell_partnered_to],cellFate[i1_]] * (z-z[cell_partnered_to])/d

                elseif tij_c[cell_partnered_to] >= tij[cell_partnered_to] || d >= ndist_P || d < 2*r

                    PI[cell_partnered_to] = 0
                    tij_c_new = 0.0
                    tij_c[cell_partnered_to] = tij_c_new
                    kp_new = CBMDistributions.uniform(0,1)
                    kp[cell_partnered_to] = kp_new

                    idx = findfirst(==(cell_partnered_to), PI_list[i1_])
                    if idx !== nothing
                        deleteat!(PI_list[i1_], idx)
                    end
                       
                end
                   
            end
        end
        #############################################################################################################################





        
        #Differentiation
        if N >= N_diff_min && diff_ON == 1
        
            rand_number = CBMDistributions.uniform(0,1)
            # AS PERCENTAGE OF NEIGHBOURS:
            if nNeighs > 0 && cellFate == 1 && rand_number < ( p*dt / (1.0 + κ0[cellFate]*nNeighs_A/nNeighs) )
                cellFate = 2
            elseif nNeighs > 0 && cellFate == 2 && rand_number < ( q*dt / (1.0 + κ0[cellFate]*nNeighs_A/nNeighs) ) 
                cellFate = 3
            end
        end

        # Colour aggregates differently if we have protrusions and N = Nmax:
        if N == Nmax && prot_ON == 1
            if cell_aggregate == 1
                cell_aggregate = 3
            end
            if cell_aggregate == 2
                cell_aggregate = 4
            end
        end



        
        #Proliferation


        if N >= N_start_sim && N_start_sim_switch == 0
            println("Starting REAL simulation, N = N_start_sim at t = ",t)
            τDiv_value = τDiv_[cellFate]
            if N_start_sim_switch == 0
                tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate]))
            end
            N_start_sim_switch = 1
        end


        if N_start_sim_switch == 0
            τDiv_value = τDiv_pre_[cellFate]
        else
            τDiv_value = τDiv_[cellFate]
        end
        
        if t > tDivision && N < Nmax
            
            if halt_division == 0
            
                #Put PI and P values to 0. (P of cell dividing will disappear after daughter cells are created).
                if PI[i1_] != 0
                    cell_partner = abs(PI[i1_])
                    PI[i1_] = 0
                    PI[cell_partner] = 0
                    tij_c = 0
                    tij_c[cell_partner] = 0
                    kp[cell_partner] = CBMDistributions.uniform(0,1)
                end
    
                #Choose random direction in unit sphere
                xₐ = CBMDistributions.normal(0,1); yₐ = CBMDistributions.normal(0,1); zₐ = CBMDistributions.normal(0,1)
                Tₐ = sqrt(xₐ^2+yₐ^2+zₐ^2)
                xₐ /= Tₐ; yₐ /= Tₐ; zₐ /= Tₐ    
    
                #Chose a random distribution of the material
                dist = CBMDistributions.uniform(1-σc,1+σc)

                rnew = r
                rsep = 0.4*r
               
                @addAgent( #Add new agent
                    x = x+rsep*xₐ,
                    y = y+rsep*yₐ,
                    z = z+rsep*zₐ,
                    
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
    
                    r = rnew,
                    c = c*(dist),
                    tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate])),
                    #############################################################################################################################
                    kp = CBMDistributions.uniform(0,1),
                    tij = 0.0,
                    cell_dividing = 1,
                    #############################################################################################################################
                )
                @addAgent( #Add new agent
                    x = x-rsep*xₐ,
                    y = y-rsep*yₐ,
                    z = z-rsep*zₐ,
                   
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
    
                    r = rnew,
                    c = c*(2-dist),
                    tDivision = t + CBMDistributions.uniform(τDiv_value*(1-σDiv_[cellFate]),τDiv_value*(1+σDiv_[cellFate])),
                    #############################################################################################################################
                    kp = CBMDistributions.uniform(0,1),
                    tij = 0.0,
                    cell_dividing = 1,
                    #############################################################################################################################
                )
                @removeAgent() # Remove agent that divided
            
            else

                halt_division = 0
                
            end
            
        end
        
    end,

    agentAlg=CBMIntegrators.Heun()
);

## Community construction and initialisation

Once with the model created, we have to construct an initial Community of agents to evolve.

### Parameters

The model from the original version has some parameters defined. We create a dictionary with all the parameters from the model assigned.

In [ ]:
parameters = Dict([
    :α => 10,
    :K => .9,
    :nn => 2,
    :mm => 2,
    :fRange => 1.2,
    :mi => 10E-6,
    :ri => 1.0,
    :b => 1.0,
    #############################################################################################################################
    :fRange_P => 4.0*sqrt(3.0),
    :λ => 1E+0,   
    :ζ => 1E+0,
    :kpON_ => [0.0 0.0 0.0; 2.0 2.0 2.0; 2.0 2.0 2.0],
    :kpOFF_ => [2.0 20.0 2.0; 20.0 0.4 20.0; 2.0 20.0 2.0],
    :τp => 0.0,
    :ε => 1.0,
    :p => 0.03,
    :q => 0.01,
    :κ0 => [0.2 0.2 0.0],
    :fs0 => 10.0,
    :fs => [0 0 0; 0 0 1; 0 1 0],
    :f0_0 => 5.0,  # multiplicative factor of repulsive part of force responsible for "cell stiffness"
    :vel_diss => 0.98, # velocity/momentum dissipation for relative velocity
    #############################################################################################################################
    :k0 => 1E+0,
    :fAdh => 1.5,
    :μ => 2.0, #this must be greater than 1 for the model to make sense!
    :τDiv_pre_ => [5 5 5],
    :τDiv_ => [120 120 120],
    :σDiv_ => [0.5 0.5 0.5],
    :c0 => 3,
    :σc => 0.01,
    :nCirc => 20,
    :σNCirc => .1,
    :fMin => .05,
    :fMax => .95,
    :fPrE => .2,
    :fEPI => .8,
    :τCirc => 45.,
    :στCirc => .02,
    :rESC => 2,
    :f0 => [0.3 0.05 0.3; 0.05 1.5 0.05; 0.3 0.05 0.3],
    :f0_rep => [0.1 0.1 0.1; 0.1 0.3 0.1; 0.1 0.1 0.1],
    :fPI_ => [5 0 5; 0 5 0; 5 0 5],
]);

### Initialise the community

The model starts from just one agent. Create the community and assign all the parameters to the Community object.

In [ ]:
function initializeEmbryo(parameters;dt,N)

    com = Community(
                model,
                N=N,
                dt=dt,
                )

    #Global parameters
    for (par,val) in pairs(parameters)
        com[par] = val
    end

    com.nOn = rand(Uniform(parameters[:nCirc]-parameters[:σNCirc],parameters[:nCirc]+parameters[:σNCirc]))
    com.cMax = parameters[:α]/(1+1/(2*parameters[:K])^(2*parameters[:mm]))

    #########Local parameters and variables###########
    com.f0 = parameters[:k0].*parameters[:f0]# / parameters[:fAdh]
    com.f0_rep = parameters[:k0].*parameters[:f0_rep]
    
    #Initialise locals
    com.m = parameters[:mi]
    com.r = parameters[:ri]
    com.cellFate = 1 #Start neutral (A) fate 
    #com.cellFate = 2 #Start with T+ fate
    com.cell_aggregate = 1 # Which aggregate cells belong to
    com.cell_dividing = 0
    com.cell_div_relax_steps = cell_div_relax_steps # This should be a function of the potential no?
    com.salt_and_pepper_switch = 0
    com.N_start_sim_switch = 0
    # start with 2 aggregates if N=2:
    if N == 2
        com.cell_aggregate[1] = 1 #Which aggregate cells belong to
        com.cell_aggregate[2] = 2
    end
    com.tOff = false #Start with the tOff deactivated
    #Initialise variables
    #############################################################################################################################
    com.nNeighs = 0 #Start with the nNeighs = 0
    com.nNeighs_A = 0 #Start with the nNeighs_A = 0
    com.nNeighs_B = 0 #Start with the nNeighs_B = 0
    com.nNeighs_C = 0 #Start with the nNeighs_C = 0
    com.nNeighs_P = 0 #Start with the nNeighs_P = 0
    com.nSumVx = 0. #Start with the nSumVx = 0.
    com.nSumVy = 0. #Start with the nSumVy = 0.
    com.nSumVz = 0. #Start with the nSumVz = 0.
    com.tij = 0. #Start with tij = 0.
    com.tij_c = 0. #Start with tij = 0.
    com.kp = CBMDistributions.uniform(0,1) #Start with random vlaues of kp drawn from uniform dist.
    #############################################################################################################################
    com.x = 0.
    if N == 2
        com.x[1] = -5.5
        com.x[2] = 5.5
    end
    com.y =  0.
    com.z =  0.
    com.vx = 0.
    com.vy = 0.
    com.vz = 0.
    com.c = com.c0
    com.tDivision = 1 #rand(Uniform(com.τDiv-com.σDiv,com.τDiv+com.σDiv))

    return com

end;

## Creating a custom evolve step

In [ ]:
function customEvolve!(com,steps,saveEach)
    loadToPlatform!(com,preallocateAgents = Nmax-N_ini+1) #loadToPlatform!(com,preallocateAgents = 100)
    #println("Nmax = ",Nmax)
    switch_0 = 0
    switch_1 = 0
    switch_2 = 0
    for i in 1:steps

        if i % 1000 == 0
            println("\n\n ***** BEGINNING NEW agentRule step ***** \n")
            println("      ***** i = ", i, " / ", steps, " and com = " , com.N, " agents *****    \n")
            switch_1 = 0
            switch_2 = 0
        end

        agentStepDE!(com)
        agentStepRule!(com)
        update!(com)
        computeNeighbors!(com)
        
        if i % saveEach == 0
            saveRAM!(com)
        end
        

        #Stop by time
        if (switch_0 == 0) && all(com.N .>= Nmax)  # if all(com.N .> 60)
            println("\n\n\n --------- REACHED THRESHHOLD! com.N = ",com.N, " REACHED THRESHHOLD! ---------")
            switch_0 = 1
        end
        
        if (switch_1 == 0) && (  any(abs.(com.vx) .> 10.0) || any(abs.(com.vy) .> 10.0) || any(abs.(com.vz) .> 10.0)  ) # if all(com.N .> 60)
            println("\n\n\n **************   WARNING!:  VELOCITY HAS REACHED THRESHOLD of 10 !!!   ************** \n")
            switch_1 = 1
        end
        if (switch_2 == 0) && (  any(abs.(com.vx) .> 100.0) || any(abs.(com.vy) .> 100.0) || any(abs.(com.vz) .> 100.0)  ) # if all(com.N .> 60)
            println("\n\n\n **************   WARNING!:  VELOCITY HAS REACHED THRESHOLDof 100 !!!   ************** \n")
            switch_2 = 2
        end
        if any(abs.(com.vx) .> 1000000.0) || any(abs.(com.vy) .> 1000000.0) || any(abs.(com.vz) .> 1000000.0) # if all(com.N .> 60)
            println("\n STOPPING BECAUSE com.vx = ",com.vx, " HAS REACHED THRESHHOLD!")
            println("STOPPING BECAUSE com.vy = ",com.vy, " IS BIGGER THAN THRESHHOLD!")
            println("STOPPING BECAUSE com.vz = ",com.vz, " IS BIGGER THAN THRESHHOLD!")
            println("  ")
            println("  STOPPED AT com.t= ",com.t, " and ", com.t/dt, " steps out of ", steps)
            println("  ")
            println("  com.fx = ",com.fx)
            println("  com.fy = ",com.fy)
            println("  com.fz = ",com.fz)
            println("  com.nSumVx = ",com.nSumVx)
            println("  com.nSumVy = ",com.nSumVy)
            println("  com.nSumVz = ",com.nSumVz)
            println("  com.fpx = ",com.fpx)
            println("  com.fpy = ",com.fpy)
            println("  com.fpz = ",com.fpz)
            println("  com.nNeighs = ",com.nNeighs)
            #println("  com.nNeighs_A = ",com.nNeighs_A)
            #println("  com.nNeighs_B = ",com.nNeighs_B)
            #println("  com.nNeighs_C = ",com.nNeighs_C)
            println("  com.nNeighs_P = ",com.nNeighs_P)
            println("  com.λ = ",com.λ)
            println("  com.ζ = ",com.ζ)
            println("  com.fRange = ",com.fRange)
            println("  com.fRange_P = ",com.fRange_P)
            println("  PI = ",PI)
            #println("  com.dt(vx) = ",com.dt(vx))
            #println("  com.dt(vy) = ",com.dt(vy))
            #println("  com.dt(vz) = ",com.dt(vz))
            break
        end
        
    end
    bringFromPlatform!(com)
    
end;

# Setup simulation

We check how the agents starts to divide and choose a fate at late stages of the simulation.

In [ ]:
random_seed = 7437
#random_seed = 12345
#random_seed = 23489
#random_seed = 99893
#random_seed = 48321
#random_seed = 5678
#random_seed = 54563
#random_seed = 19333
#random_seed = 777733
#random_seed = 6200
#random_seed = 1111
#random_seed = 44739
Random.seed!(random_seed)
#Random.seed!(91283)

rng = MersenneTwister(random_seed)  # Create an explicit RNG instance with the seed


dt = 0.01 #0.001 #0.005 #0.0005  ##0.0002  ###0.005   #small early stage tests: 0.0005

# Number of cells at start of simulation
N_ini = 1 # minimum value of 1!

# Maximum number of cells
Nmax = 8000

# Number of cells after which real simulation starts:
N_start_sim = 300

# Active cell-cell protrusions
prot_ON = 1

# When protrusions kick in
N_prot_min = N_start_sim

# Cell differentiation
diff_ON = 1
#N_diff_min = 200  #When differentiation kicks in. Can/should(?) make this stockastic.
N_diff_min = N_start_sim

# Relative velocities
N_relV_min = 1  #When relative friction kicks in.

# Number of steps/interactions required after division for cell to be considered for relative velocity calculation
cell_div_relax_steps = 50




# TO CREATE SALT & PEPPER DISTRIBUTION OF CELL FATES (ONLY WORKS WHEN STARTING WITH 100% A-TYPE AGGREGATES!):

salt_and_pepper = 0
S_and_P_B_proportion = 0.0


if salt_and_pepper == 1
    println("\n ~~~~~~~~~~~~~  MODIFYING INITIAL com.cellFate  ~~~~~~~~~~~~~ \n")
    
    
    # How many of those to convert?
    #num_to_convert = round(Int, length(indices_type1) * S_and_P_B_proportion)
    num_to_convert = round(Int, N_start_sim * S_and_P_B_proportion)
    println(" ~~~~~~~~~~~~~  HERE IS num_to_convert = ",num_to_convert, "  ~~~~~~~~~~~~~ ")
    
    # Select which ones to convert randomly
    selected_indices = randperm(rng, N_start_sim)[1:num_to_convert]  # Random indices from type 1
    println(" ~~~~~~~~~~~~~  HERE IS selected_indices = ",selected_indices, "  ~~~~~~~~~~~~~ ")
    
end


PI = zeros(Int64, Nmax)


# If we want to list the nNeighs:
nNeighs_list = zeros(Int64, 0)


nNeighs_P_list = zeros(Int64, 0)


# If we want to know which cells are bonded by prots to chosen cell:
PI_list = [Int64[] for _ in 1:Nmax]


steps = round(Int64,350/dt) #round(Int64,90/dt) #round(Int64,40/dt) #round(Int64,50/dt)  ##round(Int64,70/dt)  ##round(Int64,30/dt)     #small early stage tests: round(Int64,10/dt)
saveEach = round(Int64,1/dt) #round(Int64,0.5/dt) 





com = initializeEmbryo(parameters,dt=dt,N=N_ini);

customEvolve!(com,steps,saveEach)



# Visualization of results

We check how the agents starts to divide and choose a fate at late stages of the simulation.

In [ ]:
Colour_according_to_cell_fate = 1

In [ ]:
function getFates(com)
    d = getParameter(com,[:t,:cellFate])
    #d_2 = getParameter(com_2,[:t,:cellFate])

    dict = Dict()
    dict["t"] = [i[1] for i in d[:t]]
    dict["N"] = [length(i) for i in d[:cellFate]]
    #dict_2["t"] = [i[1] for i in d_2[:t]]
    #dict_2["N"] = [length(i) for i in d_2[:cellFate]]
    for (fateNumber, fate) in zip([1,2,3],["A","B","C"])
        dict[fate] = [sum(i.==fateNumber) for i in d[:cellFate]]
        #dict_2[fate] = [sum(i.==fateNumber) for i in d_2[:cellFate]]
    end

    
    return dict
end;

In [ ]:
colorMap = Dict("A"=>Makie.wong_colors()[1],"C"=>Makie.wong_colors()[2],"B"=>Makie.wong_colors()[3])

# DECIDE IF TO COLOUR ACCORDING TO CELL FATE OR AGGREGATE:
# FOR COLOURING ACCORDING TO CELL FATE:
if Colour_according_to_cell_fate == 1
    #colorMapNum = Dict(1=>Makie.wong_colors()[1],3=>Makie.wong_colors()[2],2=>Makie.wong_colors()[3]);
    colorMapNum = Dict(1=>(Makie.wong_colors()[1],0.2),2=>(Makie.wong_colors()[3],1.0),3=>(Makie.wong_colors()[2],0.2));
# FOR COLOURING ACCORDING TO AGGREGATE:
else  
    colorMapNum = Dict(1=>(Makie.wong_colors()[5],1.0),2=>(Makie.wong_colors()[4],1.0),3=>(Makie.wong_colors()[7],1.0),4=>(Makie.wong_colors()[6],1.0));
end
# Example with transparency!:
#colorMapNum = Dict(1=>(Makie.wong_colors()[5],0.5),2=>Makie.wong_colors()[4],3=>Makie.wong_colors()[7],4=>Makie.wong_colors()[6]);

In [ ]:
_size_ = ((maximum(com.x) - minimum(com.x)) + com.r[1]) / 1.5
_every_ = 23
#_azimuth_ = 0
labelsize = 50

fig = Figure(resolution=(5000, 4000))  # Create a larger figure

# Function to add subplots
function add_subplot(row, d, range)
    for (i, pos) in enumerate(range)
        ax = Axis3(fig[row, i], 
            xticklabelsize = labelsize,
            yticklabelsize = labelsize,
            zticklabelsize = labelsize,
            aspect = :data)

        # Corrected color mapping
        if Colour_according_to_cell_fate == 1
            color = [colorMapNum[i] for i in d[:cellFate][pos]]
        else
            color = [colorMapNum[i] for i in d[:cell_aggregate][pos]]
        end

        meshscatter!(ax, d[:x][pos], d[:y][pos], d[:z][pos], markersize=d[:r][pos], color=color)
        xlims!(ax, -_size_, _size_)
        ylims!(ax, -_size_, _size_)
        zlims!(ax, -_size_, _size_)
    end
end

# Load data
if Colour_according_to_cell_fate == 1
    d = getParameter(com, [:x, :y, :z, :r, :cellFate])
else
    d = getParameter(com, [:x, :y, :z, :r, :cell_aggregate])
end

# Add the four rows of subplots
add_subplot(1, d, 1+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4))
add_subplot(2, d, round(Int64, length(com)/4)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*2))
add_subplot(3, d, round(Int64, length(com)/4*2)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*3))
add_subplot(4, d, round(Int64, length(com)/4*3)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):length(com))

# Display and save
display(fig)
save("z_snapshots_2.png", fig)


### Make directory for saving output files 



In [ ]:
dir_name = "z_results"
mkpath(dir_name)

### Segregation score & Polarisation



In [ ]:
t0 = 44.24

In [ ]:
d2 = getParameter(com,[:x,:y,:z,:cellFate]);
d2[:cellFate];
df = DataFrame(d2);  # Convert Dict to DataFrame

In [ ]:
using DataFrames

times_expanded = Int[]
x_expanded = Float64[]
y_expanded = Float64[]
z_expanded = Float64[]
fate_expanded = Int[]

for (i, row) in enumerate(eachrow(df))  # i is timestep index
    n_cells = length(row.x)
    
    append!(times_expanded, fill(i, n_cells))  # use i as time
    
    append!(x_expanded, row.x)
    append!(y_expanded, row.y)
    append!(z_expanded, row.z)
    append!(fate_expanded, row.cellFate)
end

flat_df = DataFrame(t = times_expanded,
                    x = x_expanded,
                    y = y_expanded,
                    z = z_expanded,
                    cellFate = fate_expanded)

# Now save as CSV
using CSV
CSV.write("$dir_name/cells_flat.csv", flat_df)

#### Segregation score —— New



In [ ]:

# --- Function to compute per-cell segregation scores at a given timestep ---
function per_cell_segregation_scores(com, t; cutoff = 2.0)
    d = getParameter(com, [:x, :y, :z, :cellFate])
    x, y, z, fates = d[:x][t], d[:y][t], d[:z][t], d[:cellFate][t]
    N = length(x)

    scores = Float64[]
    fate_scores = Dict{Int, Vector{Float64}}()

    for i in 1:N
        pi = @SVector [x[i], y[i], z[i]]
        fi = fates[i]

        same_fate = 0
        total_neighbors = 0

        for j in 1:N
            if i == j
                continue
            end
            pj = @SVector [x[j], y[j], z[j]]
            if euclidean(pi, pj) < cutoff
                total_neighbors += 1
                same_fate += (fates[j] == fi) ? 1 : 0
            end
        end

        score = total_neighbors > 0 ? same_fate / total_neighbors : 0.0
        push!(scores, score)
        push!(get!(fate_scores, fi, Float64[]), score)
    end

    return scores, fate_scores
end

In [ ]:

# --- Loop over time to collect segregation scores ---
timesteps = 1:length(com)
mean_scores = Float64[]
std_scores = Float64[]

fate_scores = Dict{Int, Vector{Float64}}()
fate_stds = Dict{Int, Vector{Float64}}()

for (t_idx, t) in enumerate(timesteps)
    scores, scores_by_fate = per_cell_segregation_scores(com, t; cutoff = 2.0)

    # Whole system
    push!(mean_scores, mean(scores))
    push!(std_scores, std(scores))

    # Collect all fates seen so far
    all_fates = union(keys(fate_scores), keys(scores_by_fate))

    for fate in all_fates
        # Handle new fates by padding with NaNs
        if !haskey(fate_scores, fate)
            fate_scores[fate] = fill(NaN, t_idx - 1)
            fate_stds[fate] = fill(NaN, t_idx - 1)
        end

        if haskey(scores_by_fate, fate)
            fs = scores_by_fate[fate]
            push!(fate_scores[fate], mean(fs))
            push!(fate_stds[fate], std(fs))
        else
            push!(fate_scores[fate], NaN)
            push!(fate_stds[fate], NaN)
        end
    end
end

In [ ]:
using GLMakie, LaTeXStrings

# --- Plotting ---
sim_start_time = t0
#t0 = sim_start_time
#x_lim_min = 45
x_lim_min = t0
x_lim_max = 352
fontsize = 32

# Compute tick positions (on the original x scale)
xtick_positions = range(x_lim_min, x_lim_max; length=6)

# Set t0 as reference time: the tick closest to it should be labeled 0.0
xtick_labels = round.(((xtick_positions .- t0) .* 10 ./ 60); digits=1)

# Replace -0.0 with 0.0
xtick_labels = map(x -> x == -0.0 ? 0.0 : x, xtick_labels)

xtick_labels_latex = [latexstring(string(val)) for val in xtick_labels]

# Y-ticks, for example 0 to 1.2 in 5 steps, all LaTeXStrings
ytick_positions = range(0, 1.2; length=5)
ytick_labels_latex = [latexstring(string(round(val, digits=2))) for val in ytick_positions]

fig = Figure(resolution = (1000, 625))
ax = Axis(fig[1, 1],
    xlabel = L"Time \,\; (h)",
    ylabel = L"Segregation \,\, Score",
    xlabelsize = fontsize-2,
    ylabelsize = fontsize-2,
    titlesize = fontsize+4,
    xticks = (xtick_positions, xtick_labels_latex),
    yticks = (ytick_positions, ytick_labels_latex),
    xticklabelsize = fontsize,
    yticklabelsize = fontsize,
    xgridvisible = false,
    ygridvisible = false,
    ylabelpadding = 10,  # <-- increase this number as needed
    xlabelpadding = 10,  # <-- increase this number as needed
)

# Add empty slot in second column to add spacing
fig[1, 2] = GridLayout()  # acts like a blank spacer
colsize!(fig.layout, 2, Relative(0.05))

ylims!(ax, 0, 1.2)
xlims!(ax, x_lim_min, x_lim_max)

# Overall system curve (black)
#band!(ax, timesteps, mean_scores .- std_scores, mean_scores .+ std_scores;
#    color = (:gray, 0.3))
#lines!(ax, timesteps, mean_scores;
#    color = :black, linewidth = 2)

# Example plotting loop, make sure you define `fate_scores`, `fate_stds`, `timesteps` somewhere
colors = [:blue, :green, :orange, :yellow, :purple, :teal, :brown, :red]
color_keys = sort(collect(keys(fate_scores)))

for (i, fate) in enumerate(color_keys)
    if i == 1  # skip the first fate (blue line)
        continue
    end
    color = colors[mod1(i, length(colors))]
    fs = fate_scores[fate]
    stds = fate_stds[fate]

    band!(ax, timesteps, fs .- stds, fs .+ stds; color = (color, 0.2))
    lines!(ax, timesteps, fs; color = color, linewidth = 2, label = "Fate $fate")
end

fig
display(fig)
save("z_segregation_score_per_fate.png", fig; px_per_unit = 4)


In [ ]:
using CSV, DataFrames

combined_df = DataFrame(timestep = Int[], fate = String[], mean_score = Float64[], std_score = Float64[])

# Add global data
for (i, t) in enumerate(timesteps)
    push!(combined_df, (t, "all", mean_scores[i], std_scores[i]))
end

# Add fate-specific data
for (fate, scores) in fate_scores
    stds = fate_stds[fate]
    for (i, t) in enumerate(timesteps)
        push!(combined_df, (t, string(fate), scores[i], stds[i]))
    end
end

CSV.write("$dir_name/segregation_scores_combined.csv", combined_df)


In [ ]:
using DataFrames

# Filter combined_df to get only mean segregation scores (not std)
mean_df = combined_df[:, [:timestep, :fate, :mean_score]]

# Pivot the table so that each fate becomes a column
wide_df = unstack(mean_df, :fate, :mean_score)

# Optionally rename columns for clarity
rename!(wide_df, Dict(
    "all" => "score_all",
    "1" => "score_1",
    "2" => "score_2",
    "3" => "score_3"
))

# Sort by timestep just in case
sort!(wide_df, :timestep)

# Save to CSV (optional)
CSV.write("z_results/segregation_scores_wide.csv", wide_df)


#### Polarisation



In [ ]:
min_cluster_size = 100
cutoff = 2.0

In [ ]:
using DataFrames, LinearAlgebra, Distances, StaticArrays, Statistics

function biggest_cluster_com(
    df::DataFrame, 
    timestep::Int, 
    fate_value::Int, 
    cutoff::Float64, 
    min_cluster_size::Int = 1
)

    # 1. Extract arrays at given timestep
    cellfates = df[timestep, 1]
    x = df[timestep, 2]
    y = df[timestep, 3]
    z = df[timestep, 4]

    # 2. Compute global center of mass
    global_com = (
        x = mean(x),
        y = mean(y),
        z = mean(z)
    )

    # 3. Filter particles with desired fate
    idxs = findall(f -> f == fate_value, cellfates)
    if isempty(idxs)
        return (
            #cluster_com = (x = -2.0, y = -2.0, z = -2.0),
            cluster_com = (x = 0, y = 0, z = 0),
            #global_com = global_com,
            global_com = (x = 0, y = 0, z = 0),
            #relative_com = (x = -2.0, y = -2.0, z = -2.0),
            relative_com = (x = 0, y = 0, z = 0),
        )
    end

    # Coordinates of selected particles
    coords = [SVector(x[i], y[i], z[i]) for i in idxs]

    # 4. Cluster using cutoff
    n = length(coords)
    visited = falses(n)
    clusters = Vector{Vector{Int}}()

    for i in 1:n
        if visited[i]
            continue
        end
        neighbors = [j for j in 1:n if euclidean(coords[i], coords[j]) ≤ cutoff]
        if length(neighbors) > 1
            cluster = Set(neighbors)
            new_members = Set(neighbors)
            while !isempty(new_members)
                current = pop!(new_members)
                visited[current] = true
                next_neighbors = [j for j in 1:n if euclidean(coords[current], coords[j]) ≤ cutoff && !(j in cluster)]
                foreach(j -> push!(new_members, j), next_neighbors)
                union!(cluster, next_neighbors)
            end
            push!(clusters, collect(cluster))
        else
            visited[i] = true
        end
    end

    if isempty(clusters)
        return (
            #cluster_com = (x = -1.0, y = -1.0, z = -1.0),
            cluster_com = (x = 0.0, y = 0.0, z = 0.0),
            #global_com = global_com,
            global_com = (x = 0.0, y = 0.0, z = 0.0),
            #relative_com = (x = -1.0, y = -1.0, z = -1.0),
            relative_com = (x = 0.0, y = 0.0, z = 0.0),
        )
    end

    # 5. Find largest cluster
    largest = argmax(length.(clusters))
    cluster_indices = clusters[largest]
    
    if length(cluster_indices) < min_cluster_size
        return (
            #cluster_com = (x = -1.0, y = -1.0, z = -1.0),
            cluster_com = (x = 0.0, y = 0.0, z = 0.0),
            #global_com = global_com,
            global_com = (x = 0.0, y = 0.0, z = 0.0),
            #relative_com = (x = -1.0, y = -1.0, z = -1.0),
            relative_com = (x = 0.0, y = 0.0, z = 0.0),
        )
    end

    original_idxs = idxs[cluster_indices]
    cluster_com = (
        x = mean(x[original_idxs]),
        y = mean(y[original_idxs]),
        z = mean(z[original_idxs])
    )

    relative_com = (
        x = cluster_com.x - global_com.x,
        y = cluster_com.y - global_com.y,
        z = cluster_com.z - global_com.z
    )

    return (
        cluster_com = cluster_com,
        global_com = global_com,
        relative_com = relative_com
    )
end


In [ ]:
# FORMAT: biggest_cluster_com(df, timestep, cellFate, cutoff, Nmin):
result = biggest_cluster_com(df, 25, 1, 2.0, 20)

println("Cluster COM: ", result.cluster_com)
println("Global COM: ", result.global_com)
println("Relative COM: ", result.relative_com)


In [ ]:
using DataFrames, LinearAlgebra, Distances, StaticArrays, Statistics

function biggest_cluster_com_2(
    df::DataFrame, 
    timestep::Int, 
    fate_value::Int, 
    cutoff::Float64, 
    min_cluster_size::Int = 1
)

    # 1. Extract arrays at given timestep
    cellfates = df[timestep, 1]
    x = df[timestep, 2]
    y = df[timestep, 3]
    z = df[timestep, 4]

    # 2. Compute global center of mass
    global_com = (
        x = mean(x),
        y = mean(y),
        z = mean(z)
    )

    # 3. Filter particles with desired fate
    idxs = findall(f -> f == fate_value, cellfates)
    if isempty(idxs)
        return (
            cluster_com = (x = 0.0, y = 0.0, z = 0.0),
            global_com = global_com,
            relative_com = (x = 0.0, y = 0.0, z = 0.0),
            cluster_size = 0
        )
    end

    # Coordinates of selected particles
    coords = [SVector(x[i], y[i], z[i]) for i in idxs]

    # 4. Cluster using cutoff
    n = length(coords)
    visited = falses(n)
    clusters = Vector{Vector{Int}}()

    for i in 1:n
        if visited[i]
            continue
        end
        neighbors = [j for j in 1:n if euclidean(coords[i], coords[j]) <= cutoff]
        if length(neighbors) > 1
            cluster = Set(neighbors)
            new_members = Set(neighbors)
            while !isempty(new_members)
                current = pop!(new_members)
                visited[current] = true
                next_neighbors = [j for j in 1:n if euclidean(coords[current], coords[j]) <= cutoff && !(j in cluster)]
                foreach(j -> push!(new_members, j), next_neighbors)
                union!(cluster, next_neighbors)
            end
            push!(clusters, collect(cluster))
        else
            visited[i] = true
        end
    end

    # 5. Handle no clusters found
    if isempty(clusters)
        return (
            cluster_com = (x = 0.0, y = 0.0, z = 0.0),
            global_com = global_com,
            relative_com = (x = 0.0, y = 0.0, z = 0.0),
            cluster_size = 0
        )
    end

    # 6. Find largest cluster
    largest = argmax(length.(clusters))
    cluster_indices = clusters[largest]
    size_largest = length(cluster_indices)

    # 7. Respect min_cluster_size
    if size_largest < min_cluster_size
        return (
            cluster_com = (x = 0.0, y = 0.0, z = 0.0),
            global_com = global_com,
            relative_com = (x = 0.0, y = 0.0, z = 0.0),
            #cluster_size = 0
            cluster_size = size_largest
        )
    end

    # 8. Compute COM of the largest cluster
    original_idxs = idxs[cluster_indices]
    cluster_com = (
        x = mean(x[original_idxs]),
        y = mean(y[original_idxs]),
        z = mean(z[original_idxs])
    )

    relative_com = (
        x = cluster_com.x - global_com.x,
        y = cluster_com.y - global_com.y,
        z = cluster_com.z - global_com.z
    )

    return (
        cluster_com = cluster_com,
        global_com = global_com,
        relative_com = relative_com,
        cluster_size = size_largest
    )
end

In [ ]:
# FORMAT: biggest_cluster_com(df, timestep, cellFate, cutoff, Nmin):
result = biggest_cluster_com_2(df, 350, 2, 2.0, 300)

println("Cluster COM: ", result.cluster_com)
println("Global COM: ", result.global_com)
println("Relative COM: ", result.relative_com)
println("COM cluster size: ", result.cluster_size)

In [ ]:
function compute_com_series(df::DataFrame, fate_value::Int, cutoff::Float64, min_cluster_size::Int = 1)
    results = []

    for t in 1:size(df, 1)
        result = biggest_cluster_com(df, t, fate_value, cutoff, min_cluster_size)

        push!(results, (
            timestep = t,
            cluster_x = result.cluster_com.x,
            cluster_y = result.cluster_com.y,
            cluster_z = result.cluster_com.z,
            global_x = result.global_com.x,
            global_y = result.global_com.y,
            global_z = result.global_com.z,
            rel_x = result.relative_com.x,
            rel_y = result.relative_com.y,
            rel_z = result.relative_com.z,
            rel_dist = norm([
                result.relative_com.x,
                result.relative_com.y,
                result.relative_com.z
            ])
        ))
    end

    df_result = DataFrame(results)

    # Compute normalized norm (skip negative marker values)
    valid_norms = filter(!isnan, filter(x -> x ≥ 0, df_result.rel_dist))
    max_norm = isempty(valid_norms) ? NaN : maximum(valid_norms)

    df_result.rel_dist_norm = if isnan(max_norm) || max_norm == 0
        fill(NaN, size(df_result, 1))
    else
        [r ≥ 0 ? r / max_norm : r for r in df_result.rel_dist]
    end

    return df_result
end


In [ ]:
#min_cluster_size = 200
#cutoff = 2.0
cell_fate = 2
# FORMAT: compute_com_series(df, cellFate, cutoff, min_cluster_size):
df_com_series = compute_com_series(df, cell_fate, cutoff, min_cluster_size)

#first(df_com_series, 5)  # preview first 5 rows
last(df_com_series, 5)  # preview last 5 rows


In [ ]:
function compute_com_series_2(df::DataFrame, fate_value::Int, cutoff::Float64, min_cluster_size::Int = 1)
    results = []

    for t in 1:size(df, 1)
        result = biggest_cluster_com_2(df, t, fate_value, cutoff, min_cluster_size)

        push!(results, (
            timestep = t,
            cluster_x = result.cluster_com.x,
            cluster_y = result.cluster_com.y,
            cluster_z = result.cluster_com.z,
            global_x = result.global_com.x,
            global_y = result.global_com.y,
            global_z = result.global_com.z,
            rel_x = result.relative_com.x,
            rel_y = result.relative_com.y,
            rel_z = result.relative_com.z,
            cluster_size = result.cluster_size,
            rel_dist = norm([
                result.relative_com.x,
                result.relative_com.y,
                result.relative_com.z
            ])
        ))
    end

    df_result = DataFrame(results)

    # Compute normalized norm (skip negative marker values)
    valid_norms = filter(!isnan, filter(x -> x ≥ 0, df_result.rel_dist))
    max_norm = isempty(valid_norms) ? NaN : maximum(valid_norms)

    df_result.rel_dist_norm = if isnan(max_norm) || max_norm == 0
        fill(NaN, size(df_result, 1))
    else
        [r ≥ 0 ? r / max_norm : r for r in df_result.rel_dist]
    end

    return df_result
end

In [ ]:
#min_cluster_size = 300
#cutoff = 2.0
cell_fate = 2
# FORMAT: compute_com_series(df, cellFate, cutoff, min_cluster_size):
df_com_series = compute_com_series_2(df, cell_fate, cutoff, min_cluster_size)

#first(df_com_series, 5)  # preview first 5 rows
last(df_com_series, 5)  # preview last 5 rows

In [ ]:
using GLMakie, DataFrames, LaTeXStrings

#sim_start_time = 47.28
sim_start_time = t0
#t0 = 46.48
#x_lim_min = 45
x_lim_min = t0
x_lim_max = 352

function plot_relative_com_evolution_2(com_df::DataFrame; normalized=true, min_cluster_size=min_cluster_size, fontsize=30, padding_frac=0.05)
    rel_dist_col = normalized ? :rel_dist_norm : :rel_dist
    valid_mask = com_df[!, rel_dist_col] .> 0
    cluster_sizes = com_df.cluster_size

    min_size = isnothing(min_cluster_size) ? minimum(cluster_sizes) : min_cluster_size
    max_size = maximum(cluster_sizes)
    norm_sizes = max_size > min_size ? (cluster_sizes .- min_size) ./ (max_size - min_size) : fill(0.5, length(cluster_sizes))
    cmap = :viridis

    x_valid = com_df.timestep[valid_mask]
    y_valid = com_df[valid_mask, rel_dist_col]
    c_valid = norm_sizes[valid_mask]

    # X-axis: first and last tick exactly at data boundaries + 2 intermediate ticks
    xmin_data = minimum(x_valid)
    xmax_data = maximum(x_valid)
    xrange = xmax_data - xmin_data

    first_tick = xmin_data
    last_tick = xmax_data
    n_intermediate = 2

    intermediate_ticks = collect(range(first_tick, last_tick; length=n_intermediate + 2))[2:end-1]
    xtick_positions = vcat(first_tick, intermediate_ticks, last_tick)
    #xtick_labels = round.((xtick_positions .* 10 ./ 60); digits=1)
    #xtick_labels_latex = [latexstring(string(val)) for val in xtick_labels]
    xtick_labels = round.(((xtick_positions .- t0) .* 10 ./ 60); digits=1)
    xtick_labels_latex = [latexstring(string(val)) for val in xtick_labels]

    xmin_padded = xmin_data - padding_frac * xrange
    xmax_padded = xmax_data + padding_frac * xrange

    # Y-axis: first tick exactly zero, padding only on top
    y_min = 0
    y_max = maximum(y_valid)
    y_range = y_max - y_min
    y_max_padded = y_max + padding_frac * y_range

    n_yticks = 5
    y_tick_positions = range(y_min, y_max_padded; length=n_yticks)
    y_tick_labels_latex = [latexstring(string(round(val, digits=2))) for val in y_tick_positions]

    fig = Figure(resolution = (900, 600))
    ax = Axis(fig[1, 1],
        xlabel = L"Time \, (h)",
        ylabel = normalized ? L"Normalized \, Relative \, COM \, Distance" : L"Relative \, COM \, Distance \,\; (r)",
        xlabelsize = fontsize,
        ylabelsize = fontsize,
        titlesize = fontsize + 4,
        xticklabelsize = fontsize - 2,
        yticklabelsize = fontsize - 2,
        xticks = (xtick_positions, xtick_labels_latex),
        yticks = (y_tick_positions, y_tick_labels_latex),
        ylabelpadding = 10,  # <-- increase this number as needed
        xlabelpadding = 10,  # <-- increase this number as needed
    )
    ax.xgridvisible = false
    ax.ygridvisible = false

    xlims!(ax, xmin_padded, xmax_padded)
    ylims!(ax, y_min, y_max_padded)

    scatter!(ax, x_valid, y_valid, color = c_valid, colormap = cmap, markersize = 12)

    cb_ticks = round.(Int, range(min_size, max_size; length=3))
    cb_tick_labels_latex = [latexstring(string(val)) for val in cb_ticks]

    cb = Colorbar(fig[1, 2],
        colormap = cmap,
        limits = (min_size, max_size),
        label = L"Cluster \, Size \,\; (cells)",
        labelsize = fontsize,
        ticklabelsize = fontsize,
        ticks = (cb_ticks, cb_tick_labels_latex),
        width = 20,
        height = Relative(1),
        labelpadding = 10,  # Increase this to add more space
    )

    return fig
end


In [ ]:
fig = plot_relative_com_evolution_2(df_com_series; normalized=false, min_cluster_size)
display(fig)
save("z_colored_relative_com_evolution.png", fig)


In [ ]:
function get_largest_cluster_diameter(
    df::DataFrame, 
    fate_value::Int, 
    cutoff::Float64, 
    min_cluster_size::Int = 1;
    t_start::Int = 1  # default to the first timepoint
)
    max_cluster_diameter = 0.0
    system_diameter = 0.0

    for t in 1:size(df, 1)
        # Extract relevant particles
        cellfates = df[t, 1]
        x = df[t, 2]
        y = df[t, 3]
        z = df[t, 4]

        idxs = findall(f -> f == fate_value, cellfates)
        if isempty(idxs)
            continue
        end

        coords = [SVector(x[i], y[i], z[i]) for i in idxs]
        n = length(coords)
        visited = falses(n)
        clusters = Vector{Vector{Int}}()

        for i in 1:n
            if visited[i]
                continue
            end
            neighbors = [j for j in 1:n if euclidean(coords[i], coords[j]) ≤ cutoff]
            if length(neighbors) > 1
                cluster = Set(neighbors)
                new_members = Set(neighbors)
                while !isempty(new_members)
                    current = pop!(new_members)
                    visited[current] = true
                    next_neighbors = [j for j in 1:n if euclidean(coords[current], coords[j]) ≤ cutoff && !(j in cluster)]
                    foreach(j -> push!(new_members, j), next_neighbors)
                    union!(cluster, next_neighbors)
                end
                push!(clusters, collect(cluster))
            else
                visited[i] = true
            end
        end

        if !isempty(clusters)
            largest = argmax(length.(clusters))
            cluster_indices = clusters[largest]

            if length(cluster_indices) ≥ min_cluster_size
                points = [coords[i] for i in cluster_indices]
                d = maximum(pairwise(Euclidean(), points))
                max_cluster_diameter = max(max_cluster_diameter, d)
            end
        end
    end

    # Compute system-wide diameter at t_start
    x_all = df[t_start, 2]
    y_all = df[t_start, 3]
    z_all = df[t_start, 4]
    coords_all = [SVector(x_all[i], y_all[i], z_all[i]) for i in 1:length(x_all)]
    system_diameter = maximum(pairwise(Euclidean(), coords_all))

    return max_cluster_diameter, system_diameter
end


In [ ]:
cell_fate = 2
cluster_d, system_d = get_largest_cluster_diameter(df, cell_fate, cutoff, min_cluster_size; t_start = Int64(ceil(t0+5)))
println("Cluster diameter: ", cluster_d)
println("System diameter at t=50: ", system_d)

In [ ]:
function compute_com_series_norm_diameter(df::DataFrame, fate_value::Int, cutoff::Float64, min_cluster_size::Int = 1)
    results = []

    for t in 1:size(df, 1)
        result = biggest_cluster_com_2(df, t, fate_value, cutoff, min_cluster_size)

        push!(results, (
            timestep = t,
            cluster_x = result.cluster_com.x,
            cluster_y = result.cluster_com.y,
            cluster_z = result.cluster_com.z,
            global_x = result.global_com.x,
            global_y = result.global_com.y,
            global_z = result.global_com.z,
            rel_x = result.relative_com.x,
            rel_y = result.relative_com.y,
            rel_z = result.relative_com.z,
            cluster_size = result.cluster_size,
            rel_dist = norm([
                result.relative_com.x,
                result.relative_com.y,
                result.relative_com.z
            ])
        ))
    end

    df_result = DataFrame(results)

    # Compute normalized norm (skip negative marker values)
    valid_norms = filter(!isnan, filter(x -> x ≥ 0, df_result.rel_dist))
    max_norm = isempty(valid_norms) ? NaN : maximum(valid_norms)

    df_result.rel_dist_norm = if isnan(max_cluster_size) || max_cluster_size == 0
        fill(NaN, size(df_result, 1))
    else
        [r ≥ 0 ? r / max_cluster_size : r for r in df_result.rel_dist]
    end

    return df_result
end

In [ ]:
#min_cluster_size = 300
#cutoff = 2.0
cell_fate = 2
max_cluster_size = system_d
# FORMAT: compute_com_series(df, cellFate, cutoff, min_cluster_size):
df_com_series = compute_com_series_norm_diameter(df, cell_fate, cutoff, min_cluster_size)

#first(df_com_series, 5)  # preview first 5 rows
last(df_com_series, 5)  # preview last 5 rows

In [ ]:
#cutoff = 2.0
#min_cluster_size = 300
cell_fate_main = 2 # CellFate we are interested in
cell_fate_ref = 3 # Size of largest cluster of given cellFate to be used as normalisation factor for distance

# Compute max cluster size for fate = 3
#max_cluster_size = get_largest_cluster_diameter(df, cell_fate_ref, cutoff, min_cluster_size)
max_cluster_size = system_d

# Run normal computation
df_com_series_2 = compute_com_series_norm_diameter(df, cell_fate_main, cutoff, min_cluster_size)

# Inject normalized distances (by fate=3 cluster size)
df_com_series_2.rel_dist_norm = if isnan(max_cluster_size) || max_cluster_size == 0
    fill(NaN, size(df_com_series_2, 1))
else
    [r ≥ 0 ? r / max_cluster_size : r for r in df_com_series_2.rel_dist]
end;

last(df_com_series_2, 5)  # preview last 5 rows

In [ ]:
fig = plot_relative_com_evolution_2(df_com_series_2, normalized=true)
display(fig)
save("z_colored_normalised_relative_com_evolution.png", fig)

In [ ]:
using GLMakie, DataFrames, LaTeXStrings
#t0 = 46.48

function plot_com_with_dual_yaxes(com_df::DataFrame; 
        min_cluster_size=min_cluster_size, fontsize=30, padding_frac=0.05)

    # Columns for normalized and raw distances
    normalized_col = :rel_dist_norm
    raw_col = :rel_dist

    # Filter valid data points (only those with positive normalized distance)
    valid_mask = com_df[!, normalized_col] .> 0

    # Cluster size and color normalization
    cluster_sizes = com_df.cluster_size
    min_size = isnothing(min_cluster_size) ? minimum(cluster_sizes) : min_cluster_size
    max_size = maximum(cluster_sizes)
    norm_sizes = max_size > min_size ? (cluster_sizes .- min_size) ./ (max_size - min_size) : fill(0.5, length(cluster_sizes))
    cmap = :viridis

    # Extract valid data for plotting
    x_valid = com_df.timestep[valid_mask]
    y_norm = com_df[valid_mask, normalized_col]
    y_raw = com_df[valid_mask, raw_col]
    c_valid = norm_sizes[valid_mask]

    # X axis ticks and padding (same as original)
    xmin_data, xmax_data = minimum(x_valid), maximum(x_valid)
    xrange = xmax_data - xmin_data
    first_tick, last_tick = xmin_data, xmax_data
    intermediate_ticks = collect(range(first_tick, last_tick; length=4))[2:end-1]
    xtick_positions = vcat(first_tick, intermediate_ticks, last_tick)
    #xtick_labels = round.((xtick_positions .* 10 ./ 60); digits=1)
    xtick_labels = round.(((xtick_positions .- t0) .* 10 ./ 60); digits=1)
    xtick_labels_latex = [latexstring(string(val)) for val in xtick_labels]

    xmin_padded = xmin_data - padding_frac * xrange
    xmax_padded = xmax_data + padding_frac * xrange

    # Y axis limits and ticks for normalized values (left axis)
    y_norm_min = 0
    y_norm_max = maximum(y_norm)
    y_norm_range = y_norm_max - y_norm_min
    y_norm_max_padded = y_norm_max + padding_frac * y_norm_range
    n_yticks = 5
    y_norm_ticks = range(y_norm_min, y_norm_max_padded; length=n_yticks)
    y_norm_labels = [latexstring(string(round(val, digits=2))) for val in y_norm_ticks]

    # Y axis limits and ticks for raw values (right axis)
    y_raw_min = 0
    y_raw_max = maximum(y_raw)
    y_raw_range = y_raw_max - y_raw_min
    y_raw_max_padded = y_raw_max + padding_frac * y_raw_range
    y_raw_ticks = range(y_raw_min, y_raw_max_padded; length=n_yticks)
    y_raw_labels = [latexstring(string(round(val, digits=2))) for val in y_raw_ticks]

    # Create figure and left axis (normalized y-axis)
    fig = Figure(resolution = (900, 600))
    ax_left = Axis(fig[1, 1],
        xlabel = L"Time \, (h)",
        ylabel = L"Normalized \, Relative \, COM \, Distance",
        xlabelsize = fontsize,
        ylabelsize = fontsize,
        xticks = (xtick_positions, xtick_labels_latex),
        yticks = (y_norm_ticks, y_norm_labels),
        xticklabelsize = fontsize - 2,
        yticklabelsize = fontsize - 2,
        ylabelpadding = 10,
        xlabelpadding = 10,
    )
    ax_left.xgridvisible = false
    ax_left.ygridvisible = false
    xlims!(ax_left, xmin_padded, xmax_padded)
    ylims!(ax_left, y_norm_min, y_norm_max_padded)

    # Scatter plot on left axis
    scatter!(ax_left, x_valid, y_norm, color = c_valid, colormap = cmap, markersize = 12)

    # Create right axis sharing the x axis with ax_left
    ax_right = Axis(fig[1, 1], 
        ylabel = L"Relative \, COM \, Distance \, (r)",
        ylabelsize = fontsize,
        yticklabelsize = fontsize - 2,
        yticks = (y_raw_ticks, y_raw_labels),
        yaxisposition = :right,
        xticksvisible = false,  # Hide x ticks on right axis
        ygridvisible = false,
        backgroundcolor = (:transparent),
    )
    ax_right.xticksvisible = false
    ax_right.xticks = ([], [])
    ax_right.xlabelvisible = false
    ax_right.xlabelvisible = false
    ax_right.xgridvisible = false  # optional, if you want no grid lines on top

    
    xlims!(ax_right, xmin_padded, xmax_padded)
    ylims!(ax_right, y_raw_min, y_raw_max_padded)

    # Make right axis transparent except for labels
    hidespines!(ax_right, :l, :t, :b)  # your spine hiding as before
    ax_right.xticksvisible = false     # hide x ticks on secondary axis
    ax_right.xlabelvisible = false     # hide x label on secondary axis
    ax_right.xgridvisible = false      # hide x grid on secondary axis (optional)

    # Add colorbar (optional) on the right side outside axes
    cb_ticks = round.(Int, range(min_size, max_size; length=3))
    cb_tick_labels = [latexstring(string(val)) for val in cb_ticks]

    cb = Colorbar(fig[1, 2],
        colormap = cmap,
        limits = (min_size, max_size),
        label = L"Cluster \, Size \,\; (cells)",
        labelsize = fontsize,
        ticklabelsize = fontsize,
        ticks = (cb_ticks, cb_tick_labels),
        width = 20,
        height = Relative(1),
        labelpadding = 10,
    )

    return fig
end


In [ ]:
fig = plot_com_with_dual_yaxes(df_com_series_2)
display(fig)
save("z_com_dual_axis_plot.png", fig)


#### Save cluster COM dataframe to file



In [ ]:
CSV.write("$dir_name/cluster_com_series.csv", df_com_series_2)

In [ ]:
# Assuming you already have:
# wide_df with columns: :timestep, :score_all, :score_1, :score_2, :score_3
# df_com_series_2 with columns including :timestep, :cluster_x, :cluster_y, ... etc

# Perform an inner join by timestep
#merged_df = join(df_com_series_2, wide_df, on = :timestep, kind = :inner)
merged_df = innerjoin(df_com_series_2, wide_df, on = :timestep)

# Optionally, sort by timestep
sort!(merged_df, :timestep)

# Save merged dataframe if needed
CSV.write("z_results/merged_com_segregation.csv", merged_df)


#### Plot COM of Largest cluster of B-type cells 



In [ ]:
# DECIDE IF TO COLOUR ACCORDING TO CELL FATE OR AGGREGATE:
# FOR COLOURING ACCORDING TO CELL FATE:
if Colour_according_to_cell_fate == 1
    #colorMapNum = Dict(1=>Makie.wong_colors()[1],3=>Makie.wong_colors()[2],2=>Makie.wong_colors()[3]);
    colorMapNum = Dict(1=>(Makie.wong_colors()[1],0.1),2=>(Makie.wong_colors()[3],1.0),3=>(Makie.wong_colors()[2],0.1));
# FOR COLOURING ACCORDING TO AGGREGATE:
else  
    colorMapNum = Dict(1=>(Makie.wong_colors()[5],1.0),2=>(Makie.wong_colors()[4],1.0),3=>(Makie.wong_colors()[7],1.0),4=>(Makie.wong_colors()[6],1.0));
end

In [ ]:
#using CairoMakie
_size_ = ((maximum(com.x) - minimum(com.x)) + com.r[1]) / 1.5
_every_ = 23
_azimuth_ = 0.0
labelsize = 50

fig = Figure(resolution=(5000, 4000))  # Create a larger figure

function add_subplot(row, d, range; cellFateOfInterest::Int, cutoff::Float64 = 5.0, min_cluster_size::Int = 1)
    for (i, pos) in enumerate(range)
        ax = Axis3(fig[row, i], 
            xticklabelsize = labelsize,
            yticklabelsize = labelsize,
            zticklabelsize = labelsize,
            aspect = :data, 
            azimuth = _azimuth_ * π)

        # Color particles by cell fate or aggregate
        if Colour_according_to_cell_fate == 1
            color = [colorMapNum[i] for i in d[:cellFate][pos]]
        else
            color = [colorMapNum[i] for i in d[:cell_aggregate][pos]]
        end

        # Plot particles
        meshscatter!(ax, d[:x][pos], d[:y][pos], d[:z][pos], 
                     markersize=d[:r][pos] ./ 2.0, color=color)

        # Create a mini dataframe for this timestep
        df_t = DataFrame(
            cellFate = [d[:cellFate][pos]],
            x = [d[:x][pos]],
            y = [d[:y][pos]],
            z = [d[:z][pos]],
        )

        # Compute COM of largest cluster of given fate
        result = biggest_cluster_com(df_t, 1, cellFateOfInterest, cutoff, min_cluster_size)
        com = result.cluster_com

        # If valid COM, draw it as a red star
        #if com != (x = 0.0, y = 0.0, z = 0.0)
        #    scatter!(ax, [com.x], [com.y], [com.z], 
        #             color = :red, marker = :star5, markersize = 40)
        #end

        if com != (x = 0.0, y = 0.0, z = 0.0)
            scatter!(ax, [com.x], [com.y], [com.z]; 
            color = :red, marker = :star5, markersize = 40,
            overdraw = true, depth_shift = -1e-3)
        end


        # Set axis limits
        xlims!(ax, -_size_, _size_)
        ylims!(ax, -_size_, _size_)
        zlims!(ax, -_size_, _size_)
    end
end




# Add the four rows of subplots:

cellFateOfInterest = 2
#cutoff = 2.0
#min_cluster_size = 100

add_subplot(1, d, 1+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4); 
            cellFateOfInterest=cellFateOfInterest, cutoff=cutoff, min_cluster_size=min_cluster_size)

add_subplot(2, d, 1+round(Int64, length(com)/4)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*2); 
            cellFateOfInterest=cellFateOfInterest, cutoff=cutoff, min_cluster_size=min_cluster_size)

add_subplot(3, d, 1+round(Int64, length(com)/4*2)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*3); 
            cellFateOfInterest=cellFateOfInterest, cutoff=cutoff, min_cluster_size=min_cluster_size)

add_subplot(4, d, 1+round(Int64, length(com)/4*3)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):length(com); 
            cellFateOfInterest=cellFateOfInterest, cutoff=cutoff, min_cluster_size=min_cluster_size)

# Repeat for other rows...


# Add the four rows of subplots
#add_subplot(1, d, 1+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4))
#add_subplot(2, d, round(Int64, length(com)/4)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*2))
#add_subplot(3, d, round(Int64, length(com)/4*2)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):round(Int64, length(com)/4*3))
#add_subplot(4, d, round(Int64, length(com)/4*3)+round(Int64, length(com)/_every_):round(Int64, length(com)/_every_):length(com))

# Display and save
display(fig)
save("z_snapshots_3.png", fig)

### Make GIF of simulation 2



In [ ]:
#t0 = 46.48
function movie_frames(com, color_map, mstart, mstop;
    size = ((maximum(com.x) - minimum(com.x)) + com.r[1]) / 1.5,
    showtime = true,
    shownumbers = false,
    cellFateOfInterest::Int = 2,
    cutoff::Float64 = 2.0,
    min_cluster_size::Int = 100,
    skip::Int = 1  # New parameter: default no skipping
)
    # Build the time indices with skipping
    time_indices = collect(mstart:skip:mstop)
    m = length(string(length(time_indices)))  # for filename padding

    d = getParameter(com, Colour_according_to_cell_fate == 1 ?
        [:x, :y, :z, :r, :cellFate] :
        [:x, :y, :z, :r, :cell_aggregate]
    )
    labelsize = 30
    plots = []

    com_trail_x = Float64[]
    com_trail_y = Float64[]
    com_trail_z = Float64[]

    for (i, pos) in enumerate(time_indices)
        pos = floor(Int, pos)
        t = round(com[pos].t, digits = 2)

        fig_gif = Figure(resolution = (800, 800))
        
        t_scaled = round((com[pos].t - t0) * (10/60), digits=2)


        ax = Axis3(
            fig_gif[1, 1],
            aspect = :data,
            azimuth = _azimuth_ * π,
            elevation = _elevation_ * π,
            xlabel = "", ylabel = "", zlabel = "",
            xticklabelsize = shownumbers ? labelsize : 0,
            yticklabelsize = shownumbers ? labelsize : 0,
            zticklabelsize = shownumbers ? labelsize : 0,
            xticksvisible = false,
            yticksvisible = false,
            zticksvisible = false,
            xgridvisible = false,
            ygridvisible = false,
            zgridvisible = false,
            xspinesvisible = false,
            yspinesvisible = false,
            zspinesvisible = false,
            titlevisible = showtime,
            titlealign = :center,
            titlegap = 12,
            titlesize = labelsize,
            title = L"t = %$(t_scaled) \, h",
        )

        color = [colorMapNum[i] for i in d[Colour_according_to_cell_fate == 1 ? :cellFate : :cell_aggregate][pos]]
        meshscatter!(ax, d[:x][pos], d[:y][pos], d[:z][pos],
                     markersize = d[:r][pos]/2.0, color = color)

        # Compute COM of largest cluster for current timestep
        df_t = DataFrame(
            cellFate = [d[:cellFate][pos]],
            x = [d[:x][pos]],
            y = [d[:y][pos]],
            z = [d[:z][pos]],
        )
        result = biggest_cluster_com(df_t, 1, cellFateOfInterest, cutoff, min_cluster_size)
        com_point = result.cluster_com

        if com_point != (x = 0.0, y = 0.0, z = 0.0)
            push!(com_trail_x, com_point.x)
            push!(com_trail_y, com_point.y)
            push!(com_trail_z, com_point.z)

            lines!(ax, com_trail_x, com_trail_y, com_trail_z;
                   color = :red, linewidth = 2)

            scatter!(ax, [com_point.x], [com_point.y], [com_point.z];
                     color = :red, marker = :star5, markersize = 40,
                     overdraw = true, depth_shift = -1e-3)
        end

        xlims!(ax, -size, size)
        ylims!(ax, -size, size)
        zlims!(ax, -size, size)

        push!(plots, fig_gif)

        formatted_i = lpad(string(i), m, '0')
        save("frame_$formatted_i.png", fig_gif)
    end
end


In [ ]:
#color_map = Dict("A"=>Makie.wong_colors()[1],"C"=>Makie.wong_colors()[2],"B"=>Makie.wong_colors()[3])
color_map = Dict(1=>Makie.wong_colors()[5],2=>Makie.wong_colors()[4],3=>Makie.wong_colors()[7],4=>Makie.wong_colors()[6]);
# FOR COLOURING ACCORDING TO CELL FATE:
if Colour_according_to_cell_fate == 1
    #colorMapNum = Dict(1=>(Makie.wong_colors()[1],1.0),3=>(Makie.wong_colors()[2],1.0),2=>(Makie.wong_colors()[3],1.0));
    colorMapNum = Dict(1=>(Makie.wong_colors()[1],0.2),2=>(Makie.wong_colors()[3],1.0),3=>(Makie.wong_colors()[2],0.2));
    # FOR CAIRO MAKIE:
    #colorMapNum = Dict(1=>(Makie.wong_colors()[1],0.1),2=>(Makie.wong_colors()[3],1.0),3=>(Makie.wong_colors()[2],0.1));
# FOR COLOURING ACCORDING TO AGGREGATE:
else  
    colorMapNum = Dict(1=>(Makie.wong_colors()[5],1.0),2=>(Makie.wong_colors()[4],1.0),3=>(Makie.wong_colors()[7],1.0),4=>(Makie.wong_colors()[6],1.0));
end

_azimuth_ = 0.5
_elevation_ = 0.0
#m1 = 1;
#m1 = 1000;
#m1 = 47;
m1 = floor(t0)
#m1 = 1;
m2 = length(com);
#m2 = 50;
prop = 1
n_images = round(Int, prop * (m2 - m1))
#movie_frames(com, color_map, m1, m2, n_images)
movie_frames(
    com,
    color_map,
    m1,       # mstart
    m2,      # mstop
    skip = 1,  # Generate a frame every skip steps
    cellFateOfInterest = 2,
    cutoff = cutoff,
    min_cluster_size = min_cluster_size,
)
moment = Dates.format(now(), "dd-mm_yyyy");

In [ ]:
run(`magick "frame*".png -scale 800x800 z_movie.gif`)
#run(`magick "*".png -scale 1000x1000 -layers Optimize z_movie.gif`)
#run(`mkdir z_frames`)
#run(`mv "frame*".png 'z_frames'`)

### Proportions of aggregate types with time



In [ ]:



function get_props(com)

	d = getParameter(com, [:t, :cell_aggregate, :N])

	props = Dict()
	for state in 1:4  # 1=A, 2=B, 3=C
		props[state] = [sum(i .== state) for i in d[:cell_aggregate]]
        #println("Here is props = ", props)
        #println("Here is props[state] = ", props[state])
        #println("Here is d[:N] = ", d[:N])
		props[state] = props[state] ./ d[:N]
        #println("Here is props = ", props)
	end

	return props

end;

In [ ]:
props = get_props(com);
#println("Here is props = ", props)
#println("Here is props = ", props[2][150])

In [ ]:
function plot_proportions(com, color_map, mstart, mstop, props)

    t1 = com[mstart].t
    t2 = com[mstop].t
    
	fig = Figure(resolution = (1000, 800), figure_padding = 25)
    labelsize = 50
		ax = Axis(
        fig[1, 1],
        xlabel = "time",
        ylabel = "Proportion of cells", 
        xlabelsize = labelsize,
        ylabelsize = labelsize,
        xticklabelsize = labelsize,
        yticklabelsize = labelsize,
        aspect = 1,
        xticks = round.(range(t1, t2, 4), digits=1)
    )
	ylims!(ax, 0, 1)
	xlims!(ax, t1, t2)

	#d = getParameter(com, [:t, :cell_aggregate, :N])
    d = getParameter(com, [:t])
	plots = []
	for i in 1:4
		p = lines!(ax, d[:t][mstart:mstop], props[i][mstart:mstop], color = color_map[i], linewidth = 5) # linestyle=:dash
		push!(plots, p)
	end
	#labels = [L"state ", L"state ", L"state ", L"state "]
    labels = [" ", " ", " ", " "]
    Legend(
        fig[1, 2], 
        plots, labels, 
        labelsize = labelsize,
    )    
	display(fig)

end;


In [ ]:
props = get_props(com);      # compute proportion of each state for the saved timestamps
     
m1 = 1;
m2 = length(com);
plot_proportions(com, color_map, m1, m2, props)
# plot_proportions_analytical(com, color_map, m1, m2)
# plot_proportions_numerical_meanfield(com, color_map, m1, m2)
# plot_proportions_vs_analytical(com, color_map, m1, m2, props)
# plot_proportions_vs_meanfield(com, color_map, m1, m2, props)

### Evolution of Centre of Mass



In [ ]:
using LinearAlgebra
function get_CM(com)

	#d = getParameter(com, [:t, :cell_aggregate, :N])
    d = getParameter(com, [:t, :x, :y, :z, :N])
    #d = getParameter(com, [:x, :y, :z, :N])

    CM = Array{Float64}(undef, 3, length(com))
    CM2 = Array{Float64}(undef, 4, length(com))
    #println( "    CM = ", CM)
    #println( "    CM = ", CM[2,3])
    #CM[1] = Array{Float64}
    #CM[2] = Array{Float64}
    #CM[3] = Array{Float64}
    
	CM_ = Dict()
    #CM = d[:x]
    CM_[1] = d[:x]
    CM_[2] = d[:y]
    CM_[3] = d[:z]
    #CM[4] = d[:N]

    #propsiqs = diq[:x][11]
    
    #propsiqs = sum(diq[:x])
    
    #propsiqs[1] = sum(diq[:x])
	#for state in 1:4  # 1=A, 2=B, 3=C
    #	propsiqs[state] = [sum(i .== state) for i in d[:cell_aggregate]]
	#	propsiqs[state] = propsiqs[state] ./ d[:N]
	#end

    i = 0
    for dim in CM_
    i += 1
    j = 0
        for t in CM_[i]
            j += 1
            #println( i,"  " , j, "    CM = ", state)
            #println("OK CM = ", sum(CM[i][j]))
            ###println(sum(CM[4][j]), "    N    CM = ", sum(CM[i][j]) / sum(CM[4][j]))
            #println(d[:N][j], "    N    CM = ", sum(CM_[i][j]) / d[:N][j])
            CM[i,j] = sum(CM_[i][j]) / d[:N][j]
        end
    end

    for t in 1:length(com)
        CM2[1,t] = CM[1,t]
        CM2[2,t] = CM[2,t]
        CM2[3,t] = CM[3,t]
        CM2[4,t] = norm(CM[:,t])
    end

	return CM2

end;

In [ ]:
#println("    CM = ", CM)
#println("    CM = ", CM[1])
#println("    CM = ", CM[1][1])
CM = get_CM(com);    
#println("    CM = ", CM)
#println("    CM = ", CM[:,41])
#println("    CM = ", norm(CM[:,41]))

#println("    CM = ", CM[1][150])
#println("    CM = ", CM[3][3:101])

#println("    CM = ", CM[:,41])
#println("    size(CM) = ", size(CM))
average = mean(CM[4,:])
std_dev = std(CM[4,:])
println("    CM average = ", average)
println("    CM std_dev = ", std_dev)

In [ ]:
function plot_CM(com, color_map, mstart, mstop, CM, CM_av)

    t1 = com[mstart].t
    t2 = com[mstop].t
    
    fig = Figure(resolution = (1500, 1000), figure_padding = 25)
    labelsize = 50
    ax = Axis(
        fig[1, 1],
        xlabel = L"\text{time}",
        ylabel = L"\text{CM position}", 
        xlabelsize = labelsize,
        ylabelsize = labelsize,
        xticklabelsize = labelsize,
        yticklabelsize = labelsize,
        aspect = 1,
        xticks = round.(range(t1, t2, 4), digits=1)
    )
    ylims!(ax, minimum(CM)*1.1, maximum(CM)*1.1)
    xlims!(ax, t1, t2)

    # Extract time data
    d = getParameter(com, [:t])
    plots = []

    # Plot CM components
    for i in 1:4
        p = lines!(ax, d[:t][mstart:mstop], CM[i, mstart:mstop], color = color_map[i], linewidth = 3)
        push!(plots, p)
    end

    # Plot CM_av as a horizontal line
    av_color = :black  # Choose a color for the average plot
    av_line = lines!(ax, [t1, t2], [CM_av, CM_av], color = av_color, linewidth = 3, linestyle=:dash)
    push!(plots, av_line)

    # Add label for CM_av curve in LaTeX format
    label_x = t1 + (t2 - t1) * 0.03  # Adjust position of the label
    label_y = CM_av * 1.15
    text!(
        ax, label_x, label_y, 
        text = "<CM> = $(round(CM_av, digits=2))",  # Display CM_av value rounded to 2 decimals
        align = (:left, :center), 
        fontsize = labelsize/2,
        color = av_color
    )

    # Add labels for the legend
    labels = [L"CM_x", L"CM_y", L"CM_z", L"|CM|", L"\langle CM \rangle"]

    Legend(
        fig[1, 2], 
        plots, labels, 
        labelsize = labelsize,
    )    
    display(fig)

    # For saving figure:
     save("z_CM.png", fig)

end;


In [ ]:
color_map = Dict(1=>Makie.wong_colors()[5],2=>Makie.wong_colors()[4],3=>Makie.wong_colors()[7],4=>Makie.wong_colors()[6]);
m1 = 1;
m2 = length(com);
#println("    m1 = ", m1)
#println("    m2 = ", m2)
plot_CM(com, color_map, m1, m2, CM, average)

In [ ]:
colorMap = Dict("A"=>Makie.wong_colors()[1],"C"=>Makie.wong_colors()[2],"B"=>Makie.wong_colors()[3])
fig = Figure(resolution=(1801,901))

ax = Axis(fig[1,1],xlabel="Time",ylabel="Proportions",xlabelsize=80,ylabelsize=80,
          xticklabelsize = 40,  # Tick label size for x-axis
          yticklabelsize = 40)  # Tick label size for y-axis
fates = getFates(com)
offset = zeros(length(fates["N"]))
plots2 = []
for i in ["A","B","C"]
#for i in ["B","C"]
    prop = fates[i]./fates["N"]
    p = barplot!(ax,fates["t"],prop,offset=offset,color=colorMap[i])
    push!(plots2,p)
    offset .+= prop
end
Legend(fig[1,1], plots2, ["A","B","C"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=40)
#Legend(fig[1,1], plots2, ["B","C"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=40)

display(fig)

# For saving figure:
save("z_proportions.png", fig)

### Make statistics of the model

This model contains stochasticity in the division times and the concentration of chemical components that the daughter agents receive. This will make different runs of the simulation to differ. In order to make statistics we run the model several times and collect information of the size and fates of the cells.

In [ ]:
function makeStatistics(comBase,parameters,dt,steps,saveEach,nRepetitions)

    #Make simulations and add results to list
    d = Dict("id"=>Int64[],"N"=>Int64[],"t"=>Float64[],"A"=>Int64[],"B"=>Int64[],"C"=>Int64[])
    for i in 1:nRepetitions
        #Make the simulations
        com = initializeEmbryo(parameters,dt=dt);
        setfield!(com,:abm,comBase.abm) #Avoid world problem assigning the functions of globally declared function model
        customEvolve!(com,steps,saveEach)
    
        #Add them to the model
        fates = getFates(com)
        append!(d["id"],i*ones(Int64,length(fates["N"])))
        append!(d["N"],fates["N"])
        append!(d["t"],fates["t"])
        append!(d["A"],fates["A"])
        append!(d["B"],fates["B"])
        append!(d["C"],fates["C"])
    end

    return d
end;

In [ ]:
dt = 0.001
steps = round(Int64,50/dt)
saveEach = round(Int64,1/dt)
nRepetitions = 5

prop = makeStatistics(com,parameters,dt,steps,saveEach,nRepetitions);

In [ ]:
fig = Figure(resolution=(1000,800))

ax = Axis(fig[1,1],xlabel="N",xlabelsize=40,ylabel="proportions",ylabelsize=40)
for i in ["DP","EPI","PRE"]
    boxplot!(ax,prop["N"],prop[i]./prop["N"],color=colorMap[i])
end
Legend(fig[1,1], plots, ["DP","EPI","PRE"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=40)

ax = Axis(fig[2,1],xlabel="t",xlabelsize=40,ylabel="proportions",ylabelsize=40)
for i in ["DP","EPI","PRE"]
    boxplot!(ax,prop["t"],prop[i]./prop["N"],color=colorMap[i])
end
Legend(fig[2,1], plots, ["DP","EPI","PRE"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=40)

display(fig)

## Fitting the model

The parameters above described were chosen to match the experimental observation. This was a qualitative fitting where the parameters where tuned by hand.

In this section we will show how we can use tuning functions to choose optimize certain parameters of the model. In particular, we tune the model to fit parameters related with the chemical circuit to match the correct distributions of cells.

### Upload experimental data

We upload the experimental data that gives raise to this model.

In [ ]:
dataFull = CSV.read("data/development.csv",DataFrame)
fates = ["DP","EPI","PRE"];

In [ ]:
data = Dict("N"=>Float64[],[i=>Float64[] for i in ["DP","EPI","PRE"]]...)
for embryo in unique(dataFull[!,"Embryo_ID"])
    embryoData = dataFull[dataFull[!,"Embryo_ID"] .== embryo,:]
    push!(data["N"], 0)
    for celltype in fates
        if celltype in embryoData[!,"Identity.hc"] && celltype in ["DP","EPI","PRE"]
            val = embryoData[embryoData[!,"Identity.hc"].==celltype,"count"][1]
            push!(data[celltype],val)
            data["N"][end] += val 
        elseif celltype in ["DP","EPI","PRE"]
            push!(data[celltype],0)
        end
    end
end

In [ ]:
fig = Figure(resolution=(1000,500))

ax = Axis(fig[1,1],xlabel="N",xlabelsize=30,ylabel="Cell fates",ylabelsize=30)

offset = zeros(size(data["DP"])[1])
legend = []
order = sortperm(data["N"])
for cellId in fates
    bp = barplot!(ax,data[cellId][order], offset=offset, color = colorMap[cellId])
    push!(legend,bp)
    offset .+= data[cellId][order]
end

fig

We see that the data corresponds to sets ranging from 5 to 60 cells, being the usual sized between 5 to 30. 

In [ ]:
fig = Figure(resolution=(1000,500))
ax = Axis(fig[1,1])
legend = []

cluster = 1
for cellId in fates
    Ngrouped = round.(Int64,data["N"]/cluster).*cluster
    l = boxplot!(ax,Ngrouped,(data[cellId]./data["N"]),label=cellId, color=colorMap[cellId])
    push!(legend,l)
end
Legend(fig[1,1], legend, fates, halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=40)

fig

### Set the exploration space

The optimization algorithms require that you specify a set of parameters to optimize. in our case, our parameters correspond to parameters to the agent. However, they does not need to correcpond to parameters of the agent at all. They will be specified for the algorithm to sample from them and give new updates while optimising. 

We have to define them as a dicctionary.

In [ ]:
explore = Dict([
            :α=>(0,20),
            :K=>(0,2),
            :nn=>(0,5),
            :mm=>(0,5),
            :nCirc=>(0,30),
            :σNCirc=>(0,20),
            :c0=>(0,20)
        ]);

### Construct loos function

With the data prepared to be compared, we set the loos function. 

The loos function is a function that has to receive at least one argument, a `RowDataframe` object that contains the information of the parameters that are being fitted and has to return a value indicating how good wwere the simulations.

The function is very general so it can fit a many different routines.

Our function basically contains the following steps:

 - Sets the new parameters
 - Run several simulations for that set of parameters to get robust statistics
 - Cluster the results from the simulations as before to compare it to the experimental data
 - Compare the experimental and simulation results using a Chi Square metric as loos value.
 
The specific form of the function will depend on the optimization algorithm at hand.

In [ ]:
function loosFunction(params;parameters=parameters,data=data,nRepetitions=1,saveEach=10,dt=0.001)

    #Modify the set of parameters
    parametersModified = copy(parameters)
    parametersModified[:α] = params[:α][1]
    parametersModified[:K] = params[:K][1]
    parametersModified[:nn] = params[:nn][1]
    parametersModified[:mm] = params[:mm][1]
    parametersModified[:nCirc] = params[:nCirc][1]
    parametersModified[:σNCirc] = params[:σNCirc][1]
    parametersModified[:c0] = params[:c0][1]
    
    #Make a batch of simulations and get relevant information
    prop = makeStatistics(com,parametersModified,dt,steps,saveEach,nRepetitions);

    #Prepare data for fitting
    p = transpose([prop["DP"] prop["EPI"] prop["PRE"]]./prop["N"])
    
    #Xi square loos
    loos = 0.
    for n in minimum(data["N"]):maximum(data["N"])
        p = prop["N"] .== n
        if sum(p) > 0
            dist = [mean(prop["DP"][p])/n,mean(prop["EPI"][p])/n,mean(prop["PRE"][p])/n]
            for (nn,dp,epi,pre) in zip(data["N"],data["DP"],data["EPI"],data["PRE"])
                if nn == n
                    loos += sum((dist.-[dp,epi,pre]./n).^2)
                end
            end
        end
    end
    #Return loos
    return loos
    
end

### Check stability of loos function

We run the loos function several times to check that the results are consistent between runs. If the loos function returned different results outside the expected fluctuations, the model would not be proporly fitted as the algorithms would not be able to minimize consistently the cost.

The fluctuations for the simulations using 10 repetitions of the simulation for the same parameters show already enough consistency.

In [ ]:
#initialisation = DataFrame([:α=>parameters[:α],:K=>parameters[:K],:nn=>parameters[:nn],:mm=>parameters[:mm],:nCirc=>parameters[:nCirc],:σNCirc=>parameters[:σNCirc],:c0=>parameters[:c0]])

initialisation = DataFrame([:α=>parameters[:α],:K=>parameters[:K],:nn=>parameters[:nn],:mm=>parameters[:mm],:nCirc=>parameters[:nCirc],:σNCirc=>parameters[:σNCirc],:c0=>parameters[:c0]])

Threads.@threads for i in 1:3
    println(loosFunction(initialisation,nRepetitions=1))
end

In [ ]:
#CBMFitting.swarmAlgorithm(loosFunction,explore,population=10,stopMaxGenerations=10,saveFileName="Optimization",verbose=true)

In [ ]:
optimization = CSV.read("Optimization.csv",DataFrame);

In [ ]:
fig = Figure()
ax = Axis(fig[1,1],xticks=1:10,xlabel="Generations",xlabelsize=30,ylabel="Log loos",ylabelsize=30)

scatter!(ax,optimization._generation_.+rand(Uniform(-.2,.2),length(optimization._generation_)),(optimization._score_))
#xticks!(ax,[1,2,3],[1,2,3])

display(fig)

In [ ]:
propQualitative = makeStatistics(com,parameters,dt,steps,saveEach,nRepetitions);

In [ ]:
params = optimization[argmin(optimization[!,"_score_"]),:]
parametersModified = copy(parameters)
parametersModified[:α] = params[:α][1]
parametersModified[:K] = params[:K][1]
parametersModified[:nn] = params[:nn][1]
parametersModified[:mm] = params[:mm][1]
parametersModified[:nCirc] = params[:nCirc][1]
parametersModified[:σNCirc] = params[:σNCirc][1]
parametersModified[:c0] = params[:c0][1]

propFitted = makeStatistics(com,parametersModified,dt,steps,saveEach,10);

In [ ]:
cluster = 4
dt = 0.001
steps = round(Int64,50/dt)
saveEach = round(Int64,1/dt)
nRepetitions = 5

fig = Figure(resolution=(1500,300))

#Real
ax = Axis(fig[1,1])
legend = []
for cellId in fates
    Ngrouped = round.(Int64,data["N"]/cluster).*cluster
    l = boxplot!(ax,Ngrouped,(data[cellId]./data["N"]),label=cellId, color=colorMap[cellId])
    push!(legend,l)
end
xlims!(0,50)
Legend(fig[1,1], legend, fates, halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=20)

#Original fit
ax = Axis(fig[1,2],xlabel="N",xlabelsize=40,ylabel="proportions",ylabelsize=40)
legend = []
for i in ["DP","EPI","PRE"]
    Ngrouped = round.(Int64,propQualitative["N"]/cluster).*cluster
    l = boxplot!(ax,Ngrouped,propQualitative[i]./propQualitative["N"],color=colorMap[i])
    push!(legend,l)
end
xlims!(0,50)
Legend(fig[1,2], legend, ["DP","EPI","PRE"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=20)

#Swarm fit
ax = Axis(fig[1,3],xlabel="N",xlabelsize=40,ylabel="proportions",ylabelsize=40)
legend = []
for i in ["DP","EPI","PRE"]
    Ngrouped = round.(Int64,propFitted["N"]/cluster).*cluster
    l = boxplot!(ax,Ngrouped,propFitted[i]./propFitted["N"],color=colorMap[i])
    push!(legend,l)
end
xlims!(0,50)
Legend(fig[1,3], legend, ["DP","EPI","PRE"], halign = :right, valign = :top, tellheight = false, tellwidth = false, labelsize=20)

display(fig)